# 6 · Drawing values

**The problem.** A believable senior isn't just "name: senior": they have an age, around 70, never under 65. A pattern
needs **distributions of values**, and they too must stay rules: the same pattern checks existing contacts.

A distribution binds a **symbol** to a value drawn from it, within its body. The body's equality, `person.age == age`,
is what sets the property.

In [1]:
from mbse.Expressions import Expressions as E
from mbse.Expressions.Dialects.Python import Text
from mbse.Patterns import Constraints, Distributions, Predicates, Validators
from toolkit import Contact, Directory, book, listed

person = E.variable("person")

## Step 1: a distribution of values

`Normal`, `Uniform`, `Poisson`, `Geometric` and `Categorical` are built like any term. `.requires(...)` reads a function
whose parameters are the names it uses, outer and bound:

In [2]:
Age = (Distributions.Normal.Builder().symbol("age").mean(70).deviation(8).rounded()
       .requires(lambda person, age: person.age == age).create())
Age.symbol, Age.body

('age',
 _OperationData(name='eq', arguments=(_OperationData(name='get', arguments=(_VariableData(name='person'), _LiteralData(value='age', domain=None)), domain=None), _VariableData(name='age')), domain=None))

## Step 2: as a rule, a distribution checks its support

Checking data, a distribution takes the value its body equates the symbol with, its **witness** (`person.age`, here),
and holds when that value is in the distribution's support and the body holds. A rounded normal's support is every int,
so it checks that the age is an int. A uniform's support is its range:

In [3]:
Adults = (Predicates.OfPredicate.Builder().name("Adults").symbols({"person": Contact})
          .requires(Distributions.Uniform.Builder().symbol("age").low(18).high(64)
                    .requires(lambda person, age: person.age == age).create()).create())
store = book()
listed(store, store.Contact().name("Ann").age(34).create(), store.Contact().name("Kid").age(9).create(),
       store.Contact().name("Bob").create())
Validators.Validate(store, [Adults]).Reachable(Directory, store.singleton("book.Directory"))

["person=Contact#2: 'Adults' does not hold",
 "person=Contact#3: 'Adults' is unknown"]

The kid is outside [18, 64]. Bob has no age, so the uniform has no witness: unknown.

## Step 3: types are checked before anything runs

A distribution's values have a type, and a property has one. A normal gives floats unless it's rounded; an age is an
int. `Constraints.check` says so before any data is touched:

In [4]:
Wrong = (Predicates.OfPredicate.Builder().name("Wrong").symbols({"person": Contact})
         .requires(Distributions.Normal.Builder().symbol("age").mean(70).deviation(8)
                   .requires(lambda person, age: person.age == age).create()).create())
try:
    Constraints.check([Wrong])
except ValueError as error:
    print(error)

predicate 'Wrong': person.age is int, but its distribution gives float


## Step 4: choices of distributions

Arms can hold distributions, so a mixture is a choices. Here a senior's age is a normal restricted to 65 and over, and an
adult's is uniform:

In [5]:
HasName = (Predicates.OfPredicate.Builder().name("HasName").symbols({"person": Contact})
           .parameters(lambda p: p.name("name"))
           .requires(Text.FromFunction(lambda person, name: person.name == name)).create())
People = (Predicates.OfPredicate.Builder().name("People").symbols({"person": Contact}).requires(
    Distributions.Choices.Builder().arms(
        lambda a: a.weight(3).requires(HasName(person, "senior"), Age, person.age.ge(65)),
        lambda a: a.weight(7).requires(HasName(person, "adult"), Distributions.Uniform.Builder().symbol("age")
                                       .low(18).high(64).requires(lambda person, age: person.age == age).create()),
    ).decreasing().create()).create())
Constraints.check([People]).predicates[0].name

'People'

The next case study generates contacts from `People`.

## Why it works this way

- **Values reach properties only through equalities.** A distribution draws a value; the body says where it goes. That
  keeps the distribution a rule too: the same equality, read the other way, gives the witness a validator checks.
- **Unknown when there's no witness.** A distribution that doesn't equate its symbol with anything can't be checked
  against data, and says so.
- **Rounded, not truncated.** `.rounded()` rounds half up, so ints come out of a normal centred where you put it.